# Rotten Tomatoes pilot: audit, score normalization and join

**Midterm Project:** Curation Plan and Pilot

**Course:** ITAI 3375 - Curating DATA for A.I. Optimization

**Team members:** Chloe Tu, Kaire Campomanes

**Date:** 10/08/26


This notebook is the working pilot for the midterm curation plan. It downloads the two real Kaggle files with the Kaggle API (the kagglehub library), audits them with counts, and then tests the riskiest assumption in the plan: that mixed format review scores can be put on one 0 to 100 scale and joined to the movie table without losing or distorting many rows.

**Dataset:** https://www.kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset

**How to run it**
1. Open the notebook in Google Colab or in Jupyter with internet access. Choose Run all.
2. The first code cell downloads the dataset with `kagglehub.dataset_download`. If Kaggle asks for credentials, run `kagglehub.login()` once and then run the cell again.
3. If you cannot reach Kaggle, download the two CSV files by hand, put them in one folder, and set `LOCAL_DATA_DIR` in the first code cell. The notebook uses that folder when the download fails.
4. The same seed always gives the same sample.

The three pilot result files (cleaned data, rejects and report) contain no critic names and no review text. The one exception is `manual_audit_sample.csv`, which holds critic names and review text so the team can read the records. Keep that file private and do not upload it. The submission includes `manual_audit_log_redacted.csv` instead, which records the result of reading those 100 records without names or review text.

## 1. Setup and download

In [1]:
import glob, hashlib, json, os, re, subprocess, sys
import numpy as np
import pandas as pd

LOCAL_DATA_DIR = "data"                     # fallback folder with the two CSV files, used only if the Kaggle download fails
OUT_DIR = "pilot_output"                    # results are written here
SEED = 42
SAMPLE_N = 50000                            # size of the pilot sample
HAND_SAMPLE_N = 100                         # records exported for reading by hand
os.makedirs(OUT_DIR, exist_ok=True)

try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
    try:
        import kagglehub
    except ImportError:
        kagglehub = None

try:
    if kagglehub is None:
        raise ImportError("kagglehub is not installed")
    # Download latest version
    path = kagglehub.dataset_download("stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset")
    print("Path to dataset files:", path)
    DATA_DIR = path
    SOURCE = "Kaggle API (kagglehub)"
except Exception as e:
    print("Kaggle download did not work here:", type(e).__name__, str(e)[:150])
    print("Falling back to the local folder:", LOCAL_DATA_DIR)
    DATA_DIR = LOCAL_DATA_DIR
    SOURCE = "local folder"

def find(name):
    hits = glob.glob(f"{DATA_DIR}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {DATA_DIR}")
    return hits[0]

movies_path = find("rotten_tomatoes_movies.csv")
reviews_path = find("rotten_tomatoes_critic_reviews.csv")

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("Data source used:", SOURCE)
for p in (movies_path, reviews_path):
    print(os.path.basename(p), "|", f"{os.path.getsize(p):,} bytes", "| sha256", sha256(p))

movies = pd.read_csv(movies_path)
reviews = pd.read_csv(reviews_path)
print("movies:", movies.shape, " reviews:", reviews.shape)

Using Colab cache for faster access to the 'rotten-tomatoes-movies-and-critic-reviews-dataset' dataset.
Path to dataset files: /kaggle/input/rotten-tomatoes-movies-and-critic-reviews-dataset
Data source used: Kaggle API (kagglehub)
rotten_tomatoes_movies.csv | 17,115,750 bytes | sha256 c21dbd4849691e9b938a1bba362e374bb1aa82d5178ce7b8e06a17c14c3e9251
rotten_tomatoes_critic_reviews.csv | 226,049,903 bytes | sha256 ceaf1851d3948ca24b7a124ac3e15dbf0436666ec8673c25dd3f2ee0a5580c1f
movies: (17712, 22)  reviews: (1130017, 8)


## 2. Data profile (Section 4 of the plan)

First the shape of each table and how much is missing in each column.

In [2]:
miss_m = movies.isna().sum()
miss_r = reviews.isna().sum()
print("MOVIES: missing values per column (count and percent)")
print(pd.DataFrame({"missing": miss_m, "percent": (100 * miss_m / len(movies)).round(1)})[miss_m > 0])
print()
print("REVIEWS: missing values per column (count and percent)")
print(pd.DataFrame({"missing": miss_r, "percent": (100 * miss_r / len(reviews)).round(1)})[miss_r > 0])

MOVIES: missing values per column (count and percent)
                        missing  percent
movie_info                  321      1.8
critics_consensus          8578     48.4
genres                       19      0.1
directors                   194      1.1
authors                    1542      8.7
actors                      352      2.0
original_release_date      1166      6.6
streaming_release_date      384      2.2
runtime                     314      1.8
production_company          499      2.8
tomatometer_status           44      0.2
tomatometer_rating           44      0.2
tomatometer_count            44      0.2
audience_status             448      2.5
audience_rating             296      1.7
audience_count              297      1.7

REVIEWS: missing values per column (count and percent)
                missing  percent
critic_name       18529      1.6
review_score     305936     27.1
review_content    65806      5.8


## 3. Duplicates and dates

In [3]:
print("movies with a repeated link:", int(movies.rotten_tomatoes_link.duplicated().sum()))
print("movies with the same title and release date:", int(movies.duplicated(["movie_title", "original_release_date"]).sum()))
dups = int(reviews.duplicated().sum())
print(f"exact duplicate review rows: {dups:,} ({100*dups/len(reviews):.1f} percent of reviews)")

rd = pd.to_datetime(reviews.review_date, errors="coerce")
print("review dates: earliest", rd.min().date(), " latest", rd.max().date())
print("reviews dated 1800-01-01:", int((rd.dt.year == 1800).sum()))
print("reviews dated before 1950:", int((rd < "1950-01-01").sum()))
placeholder = int((reviews.review_date == "2000-01-01").sum())
print(f"reviews dated exactly 2000-01-01 (looks like a placeholder): {placeholder:,} ({100*placeholder/len(reviews):.2f} percent)")

movies with a repeated link: 0
movies with the same title and release date: 2
exact duplicate review rows: 119,471 (10.6 percent of reviews)
review dates: earliest 1800-01-01  latest 2020-10-29
reviews dated 1800-01-01: 83
reviews dated before 1950: 117
reviews dated exactly 2000-01-01 (looks like a placeholder): 48,019 (4.25 percent)


## 4. Score formats

The review score is the field the pilot has to fix, so this cell shows how many different forms it takes.

In [4]:
s = reviews.review_score.astype("string").str.strip()

def score_format(x):
    if pd.isna(x): return "missing"
    if re.fullmatch(r"[A-F][+-]?", x): return "letter grade"
    if re.fullmatch(r"\d+(\.\d+)?/\d+(\.\d+)?", x): return "fraction"
    if re.fullmatch(r"\d+(\.\d+)?", x): return "bare number (no scale)"
    return "other"

reviews["score_format"] = s.map(score_format)
fmt = reviews.score_format.value_counts()
print(pd.DataFrame({"reviews": fmt, "percent": (100 * fmt / len(reviews)).round(2)}))
print()
frac = s[reviews.score_format == "fraction"]
den = frac.str.split("/").str[1]
print("distinct score strings overall:", s.nunique())
print("distinct denominators among fractions:", den.nunique())
print("most common denominators:")
print(den.value_counts().head(6))
x = frac.str.split("/", expand=True).astype(float)
print("fractions with a top larger than the bottom:", int((x[0] > x[1]).sum()))
print("fractions with a bottom of zero:", int((x[1] == 0).sum()))

                        reviews  percent
score_format                            
fraction                 693561    61.38
missing                  305936    27.07
letter grade             128918    11.41
bare number (no scale)     1598     0.14
other                         4     0.00

distinct score strings overall: 814
distinct denominators among fractions: 31
most common denominators:
review_score
5      357067
4      258454
10      69440
100      6942
6        1016
20        429
Name: count, dtype: int64
fractions with a top larger than the bottom: 46
fractions with a bottom of zero: 2


## 5. Join integrity between the two tables

In [5]:
links = set(movies.rotten_tomatoes_link)
reviews["in_movies"] = reviews.rotten_tomatoes_link.isin(links)
per_movie = reviews.groupby("rotten_tomatoes_link").size()
print("reviews whose movie is not in the movies table:", int((~reviews.in_movies).sum()))
print("movies with zero reviews:", int((~movies.rotten_tomatoes_link.isin(per_movie.index)).sum()))
print("median reviews per movie:", per_movie.median())
print("movies with fewer than 10 reviews:", int((per_movie < 10).sum()), "of", len(per_movie))

mm = movies.merge(per_movie.rename("n_reviews_in_file"), left_on="rotten_tomatoes_link", right_index=True, how="left")
mm["n_reviews_in_file"] = mm.n_reviews_in_file.fillna(0)
both = mm[mm.tomatometer_count.notna()]
mismatch = int((both.tomatometer_count != both.n_reviews_in_file).sum())
print(f"movies whose tomatometer_count differs from the reviews present in the file: {mismatch:,} of {len(both):,} ({100*mismatch/len(both):.1f} percent)")

reviews whose movie is not in the movies table: 130
movies with zero reviews: 6
median reviews per movie: 30.0
movies with fewer than 10 reviews: 2891 of 17712
movies whose tomatometer_count differs from the reviews present in the file: 2,406 of 17,668 (13.6 percent)


## 6. Equity checks

Who is in the data, who is thin or absent, and whether quality is even across groups.

In [6]:
movies["year"] = pd.to_datetime(movies.original_release_date, errors="coerce").dt.year
dec = (movies.year // 10 * 10).dropna().astype(int)
print("MOVIES BY DECADE")
print(dec.value_counts().sort_index())
dated = movies.year.notna().sum()
pre1980 = int((movies.year < 1980).sum())
print(f"films before 1980: {pre1980:,} of {dated:,} dated films ({100*pre1980/dated:.1f} percent); {int(movies.year.isna().sum()):,} films have no release date")
print()
print("TOP GENRES (a film can have several)")
print(movies.genres.dropna().str.split(", ").explode().value_counts().head(10))
print()
print("CONTENT RATING"); print(movies.content_rating.value_counts())

pub = reviews.publisher_name.value_counts()
crit = reviews.critic_name.value_counts()
print()
print("top critic reviews:", int(reviews.top_critic.sum()), f"({100*reviews.top_critic.mean():.1f} percent)")
print("distinct publishers:", len(pub), "  distinct named critics:", len(crit))
print(f"top 10 publishers supply {100*pub.head(10).sum()/len(reviews):.1f} percent of reviews; top 10 critics supply {100*crit.head(10).sum()/len(reviews):.1f} percent")
small = pub[pub < 100]
print(f"publishers with fewer than 100 reviews: {len(small):,} ({100*len(small)/len(pub):.1f} percent of publishers) supplying {100*small.sum()/len(reviews):.1f} percent of reviews")
print("NOTE: the data has no language or country column, so foreign language films cannot be counted directly.")

MOVIES BY DECADE
year
1910       6
1920      65
1930     200
1940     325
1950     444
1960     601
1970     735
1980    1224
1990    2099
2000    4312
2010    6272
2020     263
Name: count, dtype: int64
films before 1980: 2,376 of 16,546 dated films (14.4 percent); 1,166 films have no release date

TOP GENRES (a film can have several)
genres
Drama                        9542
Comedy                       5674
Action & Adventure           3551
Mystery & Suspense           3534
Art House & International    2499
Documentary                  2067
Horror                       2043
Romance                      1916
Science Fiction & Fantasy    1865
Classics                     1649
Name: count, dtype: int64

CONTENT RATING
content_rating
R        6377
NR       5474
PG-13    2979
PG       2168
G         676
NC17       38
Name: count, dtype: int64

top critic reviews: 288536 (25.5 percent)
distinct publishers: 2230   distinct named critics: 11108
top 10 publishers supply 8.9 percent of reviews

In [7]:
# Is quality even across groups? Share of reviews with no score or no text, by group.
r = reviews.merge(movies[["rotten_tomatoes_link", "year", "genres"]], on="rotten_tomatoes_link", how="left")
r["score_missing"] = r.review_score.isna()
r["text_missing"] = r.review_content.isna()
r["era"] = pd.cut(r.year, [0, 1969, 1989, 1999, 2009, 2030], labels=["before 1970", "1970s and 80s", "1990s", "2000s", "2010s and later"])
size = r.publisher_name.map(pub)
r["publisher_size"] = np.where(size >= 1000, "1000 or more reviews", np.where(size >= 100, "100 to 999", "under 100"))

def by(col):
    t = r.groupby(col, observed=True).agg(reviews=("score_missing", "size"),
                                          score_missing_pct=("score_missing", lambda v: round(100 * v.mean(), 1)),
                                          text_missing_pct=("text_missing", lambda v: round(100 * v.mean(), 1)))
    return t
print("BY ERA OF THE FILM");        print(by("era"));            print()
print("BY TOP CRITIC FLAG");        print(by("top_critic"));     print()
print("BY PUBLISHER SIZE");         print(by("publisher_size")); print()
g = r.assign(genre=r.genres.str.split(", ")).explode("genre")
tg = g.groupby("genre").agg(reviews=("score_missing", "size"), score_missing_pct=("score_missing", lambda v: round(100 * v.mean(), 1)))
print("BY GENRE (genres with more than 5,000 reviews)")
print(tg[tg.reviews > 5000].sort_values("score_missing_pct"))

BY ERA OF THE FILM
                 reviews  score_missing_pct  text_missing_pct
era                                                          
before 1970        36432               30.2              19.4
1970s and 80s      51745               26.9              25.6
1990s              82008               25.8              28.1
2000s             384092               24.5               5.1
2010s and later   552510               28.5               0.3

BY TOP CRITIC FLAG
            reviews  score_missing_pct  text_missing_pct
top_critic                                              
False        841481               24.2               6.9
True         288536               35.5               2.7

BY PUBLISHER SIZE
                      reviews  score_missing_pct  text_missing_pct
publisher_size                                                    
100 to 999             292838               31.5               3.8
1000 or more reviews   807922               24.7               6.7
under 100   

## 7. Sample for reading by hand

This cell exports a random sample of records so team members can read them and count the problems. The seed is fixed, so everyone reads the same records. The cell also counts the problems that can be detected by rule, which is a check and not a replacement for reading.

The exported file contains critic names and review text, so it is for the team only and is not part of the submitted results.

**Hand reading result.** The team read all 100 records. 47 of the 100 had at least one problem: the 37 found by the rules above plus 10 found only by reading. 28 of the 100 had a problem other than a missing score. Reading found four things the rules cannot detect: review text that is only an excerpt, publisher names written in inconsistent forms, one record with two critics in the critic field, and two labels that contradict their score. Each record is logged in `manual_audit_log_redacted.csv`, which has no critic names and no review text. The full discussion is in Section 4 of the planning document.

In [8]:
hand = reviews.sample(HAND_SAMPLE_N, random_state=SEED)[
    ["rotten_tomatoes_link", "critic_name", "top_critic", "publisher_name", "review_type", "review_score", "review_date", "review_content"]]
hand.to_csv(f"{OUT_DIR}/manual_audit_sample.csv", index=False)

def site_label(x):
    """The label the site's own rule implies for a score (B minus and above, or 60 percent and above, is Fresh)."""
    if pd.isna(x): return None
    if re.fullmatch(r"[A-F][+-]?", x): return "Fresh" if x[0] in "AB" else "Rotten"
    a, b = map(float, x.split("/"))
    return "Fresh" if a / b >= 0.6 else "Rotten"

pred = hand.review_score.map(site_label)
flags = pd.DataFrame({
    "no score": hand.review_score.isna(),
    "no review text": hand.review_content.isna(),
    "no critic name": hand.critic_name.isna(),
    "placeholder date 2000-01-01": hand.review_date == "2000-01-01",
    "score contradicts Fresh/Rotten label": pred.notna() & (pred != hand.review_type),
})
print("Problems found by rule in the", HAND_SAMPLE_N, "sampled records")
print(flags.sum())
print("records with at least one problem:", int(flags.any(axis=1).sum()))
print("records with a problem other than a missing score:", int(flags.drop(columns="no score").any(axis=1).sum()))
print()
print("Next, read the records in manual_audit_sample.csv by hand and compare with the rule counts above. The team result is in the notes below this cell.")

Problems found by rule in the 100 sampled records
no score                                27
no review text                           4
no critic name                           3
placeholder date 2000-01-01              5
score contradicts Fresh/Rotten label     2
dtype: int64
records with at least one problem: 37
records with a problem other than a missing score: 12

Next, read the records in manual_audit_sample.csv by hand and compare with the rule counts above. The team result is in the notes below this cell.


## 8. The pilot: normalize scores and join

The riskiest assumption is that the mixed score formats can be put on one scale. Rules used:

* A fraction such as 3/5 becomes 60.
* A letter grade becomes a number through a mapping table. Two mappings are compared below.
* A bare number with no scale is rejected, because the scale cannot be known.
* A fraction whose top is larger than its bottom, or whose bottom is zero, is rejected.
* Nothing is silently deleted. Every failed row is written to a rejects file with its reason.

In [9]:
LETTER_SCHOOL = {"A+": 98, "A": 95, "A-": 91, "B+": 88, "B": 85, "B-": 81, "C+": 78, "C": 75,
                 "C-": 71, "D+": 68, "D": 65, "D-": 61, "F": 30}
# Second mapping: chosen so B minus and above fall at 60 or more, and C plus and below fall under 60.
LETTER_SITE = {"A+": 100, "A": 96, "A-": 92, "B+": 86, "B": 78, "B-": 70, "C+": 55, "C": 50,
               "C-": 45, "D+": 35, "D": 30, "D-": 25, "F": 10}

def normalize(x, letters):
    if pd.isna(x): return None, "missing_score"
    x = str(x).strip()
    if x in letters: return float(letters[x]), "ok"
    m = re.fullmatch(r"(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)", x)
    if m:
        n, d = float(m.group(1)), float(m.group(2))
        if d == 0: return None, "zero_denominator"
        if n > d: return None, "numerator_exceeds_denominator"
        return round(100 * n / d, 1), "ok"
    if re.fullmatch(r"\d+(?:\.\d+)?", x): return None, "bare_number_unknown_scale"
    return None, "unrecognized_format"

movies_small = movies[["rotten_tomatoes_link", "movie_title", "genres", "original_release_date", "content_rating", "year"]]
sample = reviews.sample(SAMPLE_N, random_state=SEED)[
    ["rotten_tomatoes_link", "publisher_name", "top_critic", "review_type", "review_score", "review_date"]].copy()

def run_pilot(letters, tag):
    smp = sample.copy()
    smp["score_100"], smp["status"] = zip(*smp.review_score.map(lambda v: normalize(v, letters)))
    has_movie = smp.rotten_tomatoes_link.isin(links)
    smp.loc[(smp.status == "ok") & (~has_movie), "status"] = "no_movie_match"
    ok = smp[smp.status == "ok"].merge(movies_small, on="rotten_tomatoes_link", how="left")
    ok["site_label_from_score"] = np.where(ok.score_100 >= 60, "Fresh", "Rotten")
    ok["agrees_with_label"] = ok.site_label_from_score == ok.review_type
    is_letter = ok.review_score.astype(str).str.fullmatch(r"[A-F][+-]?")
    result = {
        "letter_map": tag,
        "sample_n": len(smp),
        "status_counts": smp.status.value_counts().to_dict(),
        "kept_rows": len(ok),
        "kept_pct": round(100 * len(ok) / len(smp), 1),
        "agreement_with_label_pct": round(100 * ok.agrees_with_label.mean(), 1),
        "agreement_fractions_pct": round(100 * ok[~is_letter].agrees_with_label.mean(), 1),
        "agreement_letter_grades_pct": round(100 * ok[is_letter].agrees_with_label.mean(), 1),
    }
    # who did the filtering remove?
    smp = smp.merge(movies_small[["rotten_tomatoes_link", "year"]], on="rotten_tomatoes_link", how="left")
    smp["era"] = pd.cut(smp.year, [0, 1969, 1989, 1999, 2009, 2030], labels=["before 1970", "1970s and 80s", "1990s", "2000s", "2010s and later"])
    smp["kept"] = smp.status == "ok"
    result["kept_pct_by_era"] = smp.groupby("era", observed=True).kept.mean().mul(100).round(1).to_dict()
    result["kept_pct_by_top_critic"] = smp.groupby("top_critic").kept.mean().mul(100).round(1).to_dict()
    result["sample_rows_with_unknown_release_year"] = int(smp.year.isna().sum())
    # outputs: no critic names, no review text
    ok.drop(columns=["review_score", "site_label_from_score"]).to_csv(f"{OUT_DIR}/pilot_cleaned_{tag}.csv", index=False)
    smp[~smp.kept][["rotten_tomatoes_link", "review_score", "status"]].to_csv(f"{OUT_DIR}/pilot_rejects_{tag}.csv", index=False)
    json.dump(result, open(f"{OUT_DIR}/pilot_report_{tag}.json", "w"), indent=1, default=str)
    return result

res_school = run_pilot(LETTER_SCHOOL, "school")
print(json.dumps(res_school, indent=1, default=str))

{
 "letter_map": "school",
 "sample_n": 50000,
 "status_counts": {
  "ok": 36203,
  "missing_score": 13722,
  "bare_number_unknown_scale": 64,
  "no_movie_match": 7,
  "numerator_exceeds_denominator": 4
 },
 "kept_rows": 36203,
 "kept_pct": 72.4,
 "agreement_with_label_pct": 89.7,
 "agreement_fractions_pct": 94.4,
 "agreement_letter_grades_pct": 64.5,
 "kept_pct_by_era": {
  "before 1970": 67.9,
  "1970s and 80s": 72.7,
  "1990s": 73.3,
  "2000s": 74.8,
  "2010s and later": 71.3
 },
 "kept_pct_by_top_critic": {
  "false": 75.2,
  "true": 64.2
 },
 "sample_rows_with_unknown_release_year": 1074
}


## 9. What failed, and a second attempt

With the first letter grade mapping, agreement with the site's own Fresh or Rotten label is much lower for letter grades than for fractions. The next cell finds out why by checking the label for every letter grade across the full file.

In [10]:
g = reviews[reviews.score_format == "letter grade"]
t = g.groupby("review_score").review_type.agg(reviews="size", fresh_pct=lambda v: round(100 * (v == "Fresh").mean(), 1))
print(t.sort_index())
ab = g.review_score.str[0].isin(list("AB"))
print()
print("A or B grades labeled Rotten:", int(((g.review_type == "Rotten") & ab).sum()))
print("C, D or F grades labeled Fresh:", int(((g.review_type == "Fresh") & ~ab).sum()))
print("letter graded reviews in total:", len(g))

              reviews  fresh_pct
review_score                    
A                9482      100.0
A-              11410      100.0
B               24358      100.0
B+              19822      100.0
B-              15754      100.0
C               14329        0.0
C+              10863        0.0
C-               9449        0.0
D                6148        0.0
D+               3802        0.0
D-               1605        0.0
F                1896        0.0

A or B grades labeled Rotten: 0
C, D or F grades labeled Fresh: 0
letter graded reviews in total: 128918


In [11]:
res_site = run_pilot(LETTER_SITE, "site_aligned")
compare = pd.DataFrame({"school mapping": res_school, "site aligned mapping": res_site}).loc[
    ["kept_pct", "agreement_with_label_pct", "agreement_fractions_pct", "agreement_letter_grades_pct"]]
print(compare)
print()
print("CAUTION: letter grade agreement of 100 percent for the second mapping is circular, because the mapping was built from the label.")
print("It shows the mapping matches the label. It does not show the mapping is the right scale.")

                            school mapping site aligned mapping
kept_pct                              72.4                 72.4
agreement_with_label_pct              89.7                 95.3
agreement_fractions_pct               94.4                 94.4
agreement_letter_grades_pct           64.5                100.0

CAUTION: letter grade agreement of 100 percent for the second mapping is circular, because the mapping was built from the label.
It shows the mapping matches the label. It does not show the mapping is the right scale.


In [12]:
# Fractions also fail near the boundary: how often is the site's label different from a fixed 60 percent cutoff?
f = reviews[reviews.score_format == "fraction"].copy()
xx = f.review_score.str.split("/", expand=True).astype(float)
f["pct"] = xx[0] / xx[1] * 100
band = pd.cut(f.pct, [-1, 39, 49, 59, 69, 79, 101], labels=["0 to 39", "40 to 49", "50 to 59", "60 to 69", "70 to 79", "80 to 100"])
print((f.groupby(band, observed=True).review_type.apply(lambda v: round(100 * (v == "Fresh").mean(), 1))).rename("percent labeled Fresh"))

pct
0 to 39       0.2
40 to 49      1.2
50 to 59      6.3
60 to 69     78.6
70 to 79     99.4
80 to 100    99.8
Name: percent labeled Fresh, dtype: float64


## 10. Duplicates hid inside the sample

The full file has many duplicate reviews, but a random sample rarely contains both copies of the same row. The next cell shows how much the sample understates the problem.

In [13]:
dup_in_sample = int(sample.duplicated().sum())
print(f"duplicates visible inside the {SAMPLE_N:,} row sample: {dup_in_sample} ({100*dup_in_sample/SAMPLE_N:.2f} percent)")
print(f"duplicates in the full file: {dups:,} ({100*dups/len(reviews):.1f} percent)")
print("Lesson: remove duplicates from the full file before drawing a sample, or count them on the full file.")

duplicates visible inside the 50,000 row sample: 216 (0.43 percent)
duplicates in the full file: 119,471 (10.6 percent)
Lesson: remove duplicates from the full file before drawing a sample, or count them on the full file.


## 11. Summary of what the pilot found

* The score formats can be put on one scale, but only after a choice about letter grades, and that choice changes the result. The site treats every C and lower as Rotten, so a school style mapping is wrong for this data.
* About one in four reviews has no score at all, so any score based analysis drops those rows. The share dropped is not even across groups, as the equity checks above show.
* Sampling before removing duplicates hid a problem that affects about one in ten rows.
* Reading 100 records by hand found problems in 47 of them, 10 more than the rules alone detected. The Fresh or Rotten label looks like the critic's own verdict, so it is not a simple function of the score.
* The pilot result files in `pilot_output` (cleaned data, rejects, report) contain no critic names and no review text. Only `manual_audit_sample.csv` holds them, and it stays private. The submission uses `manual_audit_log_redacted.csv` in its place.

Deliberately left out of scope: the full set of reviews, text analysis, outside data sources, duplicate removal before sampling, and publishing a cleaned dataset.